# Интерпретируемость: Permutation Feature Importance

Этот ноутбук отвечает на вопрос: **«На какие индикаторы смотрит нейросеть при принятии решений?»**

Мы берем уже обученную модель и прогоняем её на тестовых данных. Затем мы по очереди берем каждую колонку (например, `RSI`), перемешиваем её (превращая в мусор) и снова прогоняем модель. Чем сильнее падает прибыль без этой колонки — тем важнее она для агента.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from utils.experiment_manager import get_eval_widgets, get_experiment_paths
exp_widget = get_eval_widgets()

# Автоматическое получение путей
model_path, norm_path, tensorboard_log = get_experiment_paths(exp_widget.value)

# Поддержка старых переменных в ноутбуке
model_widget = type('obj', (object,), {'value': model_path})
norm_widget = type('obj', (object,), {'value': norm_path})
MODEL_PATH = model_path
NORM_PATH = norm_path
vec_normalize_path = norm_path


Настройка путей для сохранения/загрузки моделей:


Text(value='models/ppo_prod_model', description='Model Path:', layout=Layout(width='50%'))

Text(value='models/vec_normalize_prod.pkl', description='Norm Path:', layout=Layout(width='50%'))

## 1. Загрузка Данных и Модели

In [14]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2022-01-01', interval='4h', source='bybit_futures')
fg = FeatureGenerator()
data_features = fg.transform(df)

# Откусываем тестовый кусок
TEST_SIZE = 360
test_df = data_features.iloc[-TEST_SIZE:].reset_index(drop=True)

# ==========================================
# НАСТРОЙКИ МОДЕЛИ (ВАЖНО!)
# ==========================================
USE_FRAME_STACK = True
N_STACK = 10 

try:
    # ПОМЕНЯЙТЕ ПУТЬ НА СВОЙ!
    model = PPO.load(model_widget.value) 
    vec_normalize_path = norm_widget.value 
    print("Модель успешно загружена!")
except Exception as e:
    print(f"Ошибка загрузки: {e}")

✅ All data in cache, loading from disk...
✅ HMM Model loaded from /Users/maximsinyaev/projects/trading_rl/trading_rl/models/hmm_model.pkl
Модель успешно загружена!


## 2. Функция для замера PnL

## 3. Расчет Важности Признаков (Permutation)

In [ ]:
from core.evaluation.validation import evaluate_model, calculate_permutation_importance, plot_boruta_importance

# 1. Считаем базовый PnL
print("Считаем базовый PnL (без вмешательств)...")
base_pnl = evaluate_model(model, vec_normalize_path, test_df, USE_FRAME_STACK, N_STACK, num_seeds=10)
print(f"Базовый баланс: ${base_pnl:,.2f}")

# 2. Выбираем фичи
base_features = [
    'open_over_ema_20', 'high_over_ema_20', 'low_over_ema_20', 'close_over_ema_20',
    'log_return', 'rsi_norm', 'normalized_volume',
    'macd_line_norm', 'macd_signal_norm', 'macd_hist_norm',
    'gk_volatility', 'frac_diff_norm',
    'funding_rate', 'funding_delta', 'oi_delta'
]
ema_features = ['ema_50_norm', 'ema_100_norm']
ema_diff_features = ['ema_diff_20_50', 'ema_diff_20_100', 'ema_diff_50_100']
hmm_features = [c for c in test_df.columns if 'hmm_regime' in c]

final_features = [f for f in base_features + ema_features + ema_diff_features + hmm_features if f in test_df.columns]

# 3. Расчет важности
importance_scores = calculate_permutation_importance(
    model, vec_normalize_path, test_df, final_features, base_pnl, 
    n_iterations=10, use_frame_stack=USE_FRAME_STACK, n_stack=N_STACK, num_seeds=10
)

## 4. График Важности

In [20]:
from core.evaluation.validation import plot_boruta_importance

# Рисуем крутые ящики с усами (Boruta Boxplots)
plot_boruta_importance(importance_scores)



ИНСТРУКЦИЯ:
1. Весь ЯЩИК ПРАВЕЕ нуля: Фича ПОЛЕЗНАЯ. Удалять нельзя!
2. Ящик ПЕРЕСЕКАЕТ ноль: Фича — ШУМ. Модель не уверена в её пользе.
3. Весь ЯЩИК ЛЕВЕЕ нуля: Фича ВРЕДНАЯ. Удаление этой колонки увеличит прибыль!
